### Aufgabe 1

- import 'uni_ue' data and check all tables

In [1]:
import duckdb

con = duckdb.connect(database="data/uni_ue.duckdb", read_only=False)
con.sql("SHOW TABLES").to_df()

,name
0,assistenten
1,hoeren
2,professoren
3,pruefen
4,studenten
5,voraussetzen
6,vorlesungen


#### Aufgabe 1.1

- Show all students

In [2]:
con.sql(
    """
    SELECT *
    FROM studenten
    """
)

┌────────┬──────────────┬──────────┐
│ matrnr │     name     │ semester │
│ int32  │   varchar    │  int32   │
├────────┼──────────────┼──────────┤
│  24002 │ Xenokrates   │       18 │
│  25403 │ Jonas        │       12 │
│  26120 │ Fichte       │       10 │
│  26830 │ Aristoxenos  │        8 │
│  27550 │ Schopenhauer │        6 │
│  28106 │ Carnap       │        3 │
│  29120 │ Theophrastos │        2 │
│  29555 │ Feuerbach    │        2 │
└────────┴──────────────┴──────────┘

#### Aufgabe 1.2

- The name of students, who are in 2 semester.

In [3]:
con.sql(
    """
    SELECT name
    FROM studenten
    WHERE semester = 2
    """
)

┌──────────────┐
│     name     │
│   varchar    │
├──────────────┤
│ Theophrastos │
│ Feuerbach    │
└──────────────┘

#### Aufgabe 1.3

- The name of students and Vorlesungsnummern, which they attend.
- Students who don't take any lecture, should not appear in the result.

In [4]:
# check vorlesungen and hoeren table

con.sql(
    """
    SELECT *
    FROM vorlesungen, hoeren
    """
)

┌────────┬──────────────────┬───────┬────────────┬────────┬────────┐
│ vorlnr │      titel       │  sws  │ gelesenvon │ matrnr │ vorlnr │
│ int32  │     varchar      │ int32 │   int32    │ int32  │ int32  │
├────────┼──────────────────┼───────┼────────────┼────────┼────────┤
│   4052 │ Logik            │     4 │       2125 │  26120 │   5001 │
│   4052 │ Logik            │     4 │       2125 │  27550 │   5001 │
│   4052 │ Logik            │     4 │       2125 │  27550 │   4052 │
│   4052 │ Logik            │     4 │       2125 │  28106 │   5041 │
│   4052 │ Logik            │     4 │       2125 │  28106 │   5052 │
│   4052 │ Logik            │     4 │       2125 │  28106 │   5216 │
│   4052 │ Logik            │     4 │       2125 │  28106 │   5259 │
│   4052 │ Logik            │     4 │       2125 │  29120 │   5001 │
│   4052 │ Logik            │     4 │       2125 │  29120 │   5041 │
│   4052 │ Logik            │     4 │       2125 │  29120 │   5049 │
│     ·  │   ·              │     

In [5]:
con.sql(
    """
    SELECT s.name, h.vorlnr
    FROM Studenten s, hoeren h
    WHERE s.matrnr = h.matrnr
    """
)

┌──────────────┬────────┐
│     name     │ vorlnr │
│   varchar    │ int32  │
├──────────────┼────────┤
│ Fichte       │   5001 │
│ Schopenhauer │   5001 │
│ Schopenhauer │   4052 │
│ Carnap       │   5041 │
│ Carnap       │   5052 │
│ Carnap       │   5216 │
│ Carnap       │   5259 │
│ Theophrastos │   5001 │
│ Theophrastos │   5041 │
│ Theophrastos │   5049 │
│ Feuerbach    │   5022 │
│ Jonas        │   5022 │
│ Feuerbach    │   5001 │
└──────────────┴────────┘
  13 rows     2 columns

#### Aufgabe 1.4

- The name of all students, who don't take lecture.

In [6]:
con.sql(
    """
    SELECT name
    FROM studenten
    WHERE matrnr NOT IN (SELECT matrnr FROM hoeren)
    """
)

┌─────────────┐
│    name     │
│   varchar   │
├─────────────┤
│ Xenokrates  │
│ Aristoxenos │
└─────────────┘

#### Aufgabe 1.5

- The name of students and professors.

In [7]:
con.sql(
    """
    SELECT name
    FROM studenten
    
    union
    
    SELECT name
    FROM professoren
    """
)

┌──────────────┐
│     name     │
│   varchar    │
├──────────────┤
│ Popper       │
│ Xenokrates   │
│ Curie        │
│ Russel       │
│ Carnap       │
│ Aristoxenos  │
│ Sokrates     │
│ Feuerbach    │
│ Schopenhauer │
│ Fichte       │
│ Kopernikus   │
│ Augustinus   │
│ Jonas        │
│ Theophrastos │
│ Kant         │
└──────────────┘
    15 rows   

In [8]:
# JOIN -> can not use name in SELECT, because there are two name columns

con.sql(
    """
    SELECT s.name name_student, p.name name_professor
    FROM studenten s, professoren p
    """
)

┌──────────────┬────────────────┐
│ name_student │ name_professor │
│   varchar    │    varchar     │
├──────────────┼────────────────┤
│ Xenokrates   │ Sokrates       │
│ Jonas        │ Sokrates       │
│ Fichte       │ Sokrates       │
│ Aristoxenos  │ Sokrates       │
│ Schopenhauer │ Sokrates       │
│ Carnap       │ Sokrates       │
│ Theophrastos │ Sokrates       │
│ Feuerbach    │ Sokrates       │
│ Xenokrates   │ Russel         │
│ Jonas        │ Russel         │
│   ·          │   ·            │
│   ·          │   ·            │
│   ·          │   ·            │
│ Theophrastos │ Curie          │
│ Feuerbach    │ Curie          │
│ Xenokrates   │ Kant           │
│ Jonas        │ Kant           │
│ Fichte       │ Kant           │
│ Aristoxenos  │ Kant           │
│ Schopenhauer │ Kant           │
│ Carnap       │ Kant           │
│ Theophrastos │ Kant           │
│ Feuerbach    │ Kant           │
└──────────────┴────────────────┘
  56 rows (20 shown)  2 columns

#### Aufgabe 1.6

- The Matrikelnummern of students and number of lecture they take.
- Show the students, which take at least 3 lecture.

In [9]:
con.sql(
    """
    SELECT s.matrnr, COUNT(h.vorlnr) count
    FROM studenten s JOIN hoeren h ON s.matrnr = h.matrnr
    GROUP BY s.matrnr
    HAVING count >= 3
    """
)

┌────────┬───────┐
│ matrnr │ count │
│ int32  │ int64 │
├────────┼───────┤
│  29120 │     3 │
│  28106 │     4 │
└────────┴───────┘

In [10]:
con.close()

### Aufgabe 2

- import 'wettfahrt_ue' data and show tables

In [11]:
con = duckdb.connect(database="data/wettfahrt_ue.duckdb", read_only=False)

con.sql("SHOW TABLES").to_df()

,name
0,Bootsklasse
1,Platzierung
2,Teilnehmer
3,Wettfahrt


#### Aufgabe 2.1

- The name of Regatten and Boote that won the competition

In [12]:
con.sql(
    """
    SELECT *
    FROM Wettfahrt
    """
)


┌─────────┬──────────────────────────┬────────────┬──────────┐
│ FahrtNr │           Name           │   Datum    │   Zeit   │
│  int16  │         varchar          │    date    │   time   │
├─────────┼──────────────────────────┼────────────┼──────────┤
│       1 │ Moorpokal                │ 2003-06-18 │ 10:00:00 │
│       2 │ Herbstmeister            │ 2003-09-16 │ 14:00:00 │
│       3 │ Franz Huber Gedenk Preis │ 2003-05-15 │ 14:00:00 │
│       4 │ Blaues Band              │ 2003-05-29 │ 10:00:00 │
└─────────┴──────────────────────────┴────────────┴──────────┘

In [13]:
con.sql(
    """
    SELECT *
    FROM Platzierung
    """
)

┌──────────┬───────────┬───────┐
│ SEGELNR  │ WETTFAHRT │ platz │
│ varchar  │   int16   │ int64 │
├──────────┼───────────┼───────┤
│ GER 1393 │         3 │     1 │
│ GER 3876 │         3 │     4 │
│ GER 4309 │         3 │     3 │
│ GER 4318 │         1 │     1 │
│ GER 4318 │         2 │     2 │
│ GER 4833 │         3 │  NULL │
│ GER 4995 │         1 │     2 │
│ GER 4995 │         2 │     1 │
│ GER 5107 │         4 │     1 │
│ GER 5503 │         3 │     2 │
│ GER 5505 │         4 │     3 │
│ GER 5703 │         4 │     2 │
└──────────┴───────────┴───────┘
  12 rows            3 columns

In [14]:
con.sql(
    """
    SELECT *
    FROM Teilnehmer
    """
)

┌──────────┬───────────┬─────────────┬─────────┬─────────┬───────────┐
│ SegelNr  │   Name    │ Bootsklasse │ Baujahr │  Farbe  │  Eigner   │
│ varchar  │  varchar  │   varchar   │ varchar │ varchar │  varchar  │
├──────────┼───────────┼─────────────┼─────────┼─────────┼───────────┤
│ GER 1393 │ Carla F.  │ Folkeboot   │ 1972    │ Weiß    │ G.Gerhard │
│ GER 3876 │ No. Uno   │ Folkeboot   │ 1993    │ Rot     │ N. Nichts │
│ GER 3999 │ Willi     │ Optimist    │ 1989    │ Weiß    │ E. Ernst  │
│ GER 4309 │ Elkche    │ H-Boot      │ 1981    │ Blau    │ M. Michel │
│ GER 4318 │ Marie     │ Pirat       │ 1992    │ Blau    │ D. Dummer │
│ GER 4833 │ Martha H. │ H-Boot      │ 1994    │ Weiß    │ O. Otter  │
│ GER 4995 │ Celeste   │ Pirat       │ 1991    │ Rot     │ S. Schott │
│ GER 5107 │ Windrose  │ Optimist    │ 1987    │ Lila    │ V. Voelz  │
│ GER 5503 │ Lisa      │ H-Boot      │ 1983    │ Grün    │ H. Hiller │
│ GER 5505 │ Pistensau │ Optimist    │ 1993    │ Braun   │ F. Faser  │
│ GER 

In [15]:
con.sql(
    """
    SELECT t.name, w.name
    FROM Teilnehmer t JOIN Platzierung p ON t.SegelNr = p.SegelNr
                      JOIN Wettfahrt w ON p.WETTFAHRT = w.FahrtNr
    WHERE platz = 1
    """
)

┌──────────┬──────────────────────────┐
│   Name   │           Name           │
│ varchar  │         varchar          │
├──────────┼──────────────────────────┤
│ Carla F. │ Franz Huber Gedenk Preis │
│ Marie    │ Moorpokal                │
│ Celeste  │ Herbstmeister            │
│ Windrose │ Blaues Band              │
└──────────┴──────────────────────────┘

#### Aufgabe 2.2

- The most common boat color and SegelNr of the boats painted or vanished in that color.

In [16]:
# If the data set is small, first check with count and using HAVING

con.sql(
    """
    SELECT Farbe, COUNT(Farbe) count
    FROM Teilnehmer
    GROUP BY Farbe
    HAVING count = 3
    """
)

┌─────────┬───────┐
│  Farbe  │ count │
│ varchar │ int64 │
├─────────┼───────┤
│ Weiß    │     3 │
└─────────┴───────┘

In [17]:
# But if a data set is large, we have to use complex query

con.sql(
    """
    WITH counts AS (
        SELECT Farbe, COUNT(Farbe) count
        FROM Teilnehmer
        GROUP BY Farbe
    )
    SELECT * 
    FROM counts
    WHERE count = (SELECT MAX(count) FROM counts)
    """
)

┌─────────┬───────┐
│  Farbe  │ count │
│ varchar │ int64 │
├─────────┼───────┤
│ Weiß    │     3 │
└─────────┴───────┘